In [1]:
import pandas as pd
import numpy as np

base = "/home/darshani/lightkurve-env/space-debris-detector/data"

tle = pd.read_parquet(f"{base}/cleaned/tle_cleaned.parquet")
satcat = pd.read_parquet(f"{base}/cleaned/satcat_cleaned.parquet")
decay = pd.read_parquet(f"{base}/cleaned/decay_cleaned.parquet")
conj = pd.read_parquet(f"{base}/cleaned/conjunctions_cleaned.parquet")
discos = pd.read_parquet(f"{base}/cleaned/discos_cleaned.parquet")
sgp4 = pd.read_parquet(f"{base}/sgp4_positions.parquet")

print(tle.shape, satcat.shape, decay.shape, conj.shape, discos.shape, sgp4.shape)

(66666, 40) (67526, 24) (132699, 13) (3096, 16) (88998, 31) (37843, 12)


In [2]:
for name, df in [('TLE', tle), ('SATCAT', satcat), ('DECAY', decay), ('CONJ', conj), ('DISCOS', discos), ('SGP4', sgp4)]:
    print(f"\n{name}:", df.columns.tolist())


TLE: ['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE', 'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2']

SATCAT: ['INTLDES', 'NORAD_CAT_ID', 'OBJECT_TYPE', 'SATNAME', 'COUNTRY', 'LAUNCH', 'SITE', 'DECAY', 'PERIOD', 'INCLINATION', 'APOGEE', 'PERIGEE', 'COMMENT', 'COMMENTCODE', 'RCSVALUE', 'RCS_SIZE', 'FILE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_NAME', 'OBJECT_ID', 'OBJECT_NUMBER']

DECAY: ['NORAD_CAT_ID', 'OBJECT_NUMBER', 'OBJECT_NAME', 'INTLDES', 'OBJECT_ID', 'RCS', 'RCS_SI

In [3]:
for df in [tle, satcat, decay, sgp4]:
    df['NORAD_CAT_ID'] = pd.to_numeric(df['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# discos uses attr_satno instead of NORAD_CAT_ID
discos = discos.rename(columns={'attr_satno': 'NORAD_CAT_ID'})
discos['NORAD_CAT_ID'] = pd.to_numeric(discos['NORAD_CAT_ID'], errors='coerce').astype('Int64')

# conj has two objects per row
conj['SAT_1_ID'] = pd.to_numeric(conj['SAT_1_ID'], errors='coerce').astype('Int64')
conj['SAT_2_ID'] = pd.to_numeric(conj['SAT_2_ID'], errors='coerce').astype('Int64')

print("ids fixed!!")

ids fixed!!


In [4]:
decay = decay.sort_values('MSG_EPOCH').groupby('NORAD_CAT_ID', as_index=False).last()
print("decay:", decay.shape)

decay: (35706, 13)


In [5]:
conj['PC'] = pd.to_numeric(conj['PC'], errors='coerce')
conj['MIN_RNG'] = pd.to_numeric(conj['MIN_RNG'], errors='coerce')

sat1 = conj[['SAT_1_ID', 'PC', 'MIN_RNG']].rename(columns={'SAT_1_ID': 'NORAD_CAT_ID'})
sat2 = conj[['SAT_2_ID', 'PC', 'MIN_RNG']].rename(columns={'SAT_2_ID': 'NORAD_CAT_ID'})
conj_agg = pd.concat([sat1, sat2]).groupby('NORAD_CAT_ID').agg(
    conj_count=('PC', 'count'),
    max_pc=('PC', 'max'),
    mean_pc=('PC', 'mean'),
    min_miss_dist=('MIN_RNG', 'min')
).reset_index()

print("conj:", conj_agg.shape)

conj: (938, 5)


In [6]:
merged = tle.copy()
print("start:", merged.shape)

merged = merged.merge(satcat, on='NORAD_CAT_ID', how='left', suffixes=('', '_satcat'))
print("after satcat:", merged.shape)

merged = merged.merge(decay, on='NORAD_CAT_ID', how='left', suffixes=('', '_decay'))
print("after decay:", merged.shape)

merged = merged.merge(conj_agg, on='NORAD_CAT_ID', how='left')
print("after conj:", merged.shape)

merged = merged.merge(discos, on='NORAD_CAT_ID', how='left', suffixes=('', '_discos'))
print("after discos:", merged.shape)

merged = merged.merge(sgp4, on='NORAD_CAT_ID', how='left')
print("after sgp4:", merged.shape)

start: (66666, 40)
after satcat: (66666, 63)
after decay: (66666, 75)
after conj: (66666, 79)
after discos: (66666, 109)
after sgp4: (66666, 120)


In [7]:
merged.to_parquet(f"{base}/merged/all_merged.parquet", index=False)
print("done!!", merged.shape)
merged.head()

done!! (66666, 120)


,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T16:16:27,18 SPCS,OPS 3674 (VELA 4),1964-040B,EARTH,TEME,UTC,SGP4,...,-87842.248216,66031.086790,40166.139083,-1.098975,-0.450464,1.326665,-4.0,110631.921857,1.780647,GEO
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T16:16:27,18 SPCS,YZ-1 R/B,2015-019C,EARTH,TEME,UTC,SGP4,...,-71978.947246,26862.548852,13005.642656,-2.142409,-1.165972,0.501345,-4.0,71550.191715,2.490131,GEO
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T04:49:44,18 SPCS,ASTRON,1983-020A,EARTH,TEME,UTC,SGP4,...,-56540.136904,-127218.270086,-93797.684633,0.275027,0.036663,-0.999533,-3.0,161495.854880,1.037328,GEO
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-16T12:26:49,18 SPCS,OPS 3662 (VELA 3),1964-040A,EARTH,TEME,UTC,SGP4,...,-69574.444483,-75544.686826,128140.723117,0.671024,-0.859229,-0.177731,-3.0,157847.293602,1.104598,GEO
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T04:49:44,18 SPCS,CXO,1999-040B,EARTH,TEME,UTC,SGP4,...,-25178.252625,-94524.345224,96227.317779,0.448764,-0.820913,0.023654,-2.0,130845.955703,0.935867,GEO


In [9]:
merged = pd.read_parquet("/home/darshani/lightkurve-env/space-debris-detector/data/merged/all_merged.parquet")

# find duplicate columns (the _satcat, _decay, _discos suffixed ones)
dup_cols = [c for c in merged.columns if c.endswith('_satcat') or c.endswith('_decay') or c.endswith('_discos')]
print("duplicate cols to drop:", dup_cols)
print("shape before:", merged.shape)

duplicate cols to drop: ['OBJECT_TYPE_satcat', 'SITE_satcat', 'PERIOD_satcat', 'INCLINATION_satcat', 'COMMENT_satcat', 'RCS_SIZE_satcat', 'FILE_satcat', 'OBJECT_NAME_satcat', 'OBJECT_ID_satcat', 'OBJECT_NUMBER_decay', 'OBJECT_NAME_decay', 'INTLDES_decay', 'OBJECT_ID_decay', 'RCS_SIZE_decay', 'COUNTRY_decay']
shape before: (66666, 120)


In [10]:
dup_cols = ['OBJECT_TYPE_satcat', 'SITE_satcat', 'PERIOD_satcat', 'INCLINATION_satcat', 
            'COMMENT_satcat', 'RCS_SIZE_satcat', 'FILE_satcat', 'OBJECT_NAME_satcat', 
            'OBJECT_ID_satcat', 'OBJECT_NUMBER_decay', 'OBJECT_NAME_decay', 'INTLDES_decay', 
            'OBJECT_ID_decay', 'RCS_SIZE_decay', 'COUNTRY_decay']

merged = merged.drop(columns=dup_cols)
print("shape after dropping dupes:", merged.shape)

shape after dropping dupes: (66666, 105)


In [11]:
merged.to_parquet(f"{base}/merged/all_cleaned_merged.parquet", index=False)
print("saved!!", merged.shape)
merged.head()

saved!! (66666, 105)


,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T16:16:27,18 SPCS,OPS 3674 (VELA 4),1964-040B,EARTH,TEME,UTC,SGP4,...,-87842.248216,66031.086790,40166.139083,-1.098975,-0.450464,1.326665,-4.0,110631.921857,1.780647,GEO
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T16:16:27,18 SPCS,YZ-1 R/B,2015-019C,EARTH,TEME,UTC,SGP4,...,-71978.947246,26862.548852,13005.642656,-2.142409,-1.165972,0.501345,-4.0,71550.191715,2.490131,GEO
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T04:49:44,18 SPCS,ASTRON,1983-020A,EARTH,TEME,UTC,SGP4,...,-56540.136904,-127218.270086,-93797.684633,0.275027,0.036663,-0.999533,-3.0,161495.854880,1.037328,GEO
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-16T12:26:49,18 SPCS,OPS 3662 (VELA 3),1964-040A,EARTH,TEME,UTC,SGP4,...,-69574.444483,-75544.686826,128140.723117,0.671024,-0.859229,-0.177731,-3.0,157847.293602,1.104598,GEO
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2026-03-18T04:49:44,18 SPCS,CXO,1999-040B,EARTH,TEME,UTC,SGP4,...,-25178.252625,-94524.345224,96227.317779,0.448764,-0.820913,0.023654,-2.0,130845.955703,0.935867,GEO


In [12]:
print(merged.columns.tolist())

['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR', 'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS', 'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE', 'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2', 'INTLDES', 'SATNAME', 'COUNTRY', 'LAUNCH', 'DECAY', 'APOGEE', 'PERIGEE', 'COMMENTCODE', 'RCSVALUE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_NUMBER', 'RCS', 'MSG_EPOCH', 'DECAY_EPOCH', 'SOURCE', 'MSG_TYPE', 'PRECEDENCE', 'conj_count', 'max_pc', 'mean_pc', 'min_miss_dist', 'type', 'id', 'attr_cosparId', 'attr_vimpelId', 'attr_name', 'attr_objectClass', 'attr_mass', 'attr_shape', 'at

In [14]:
# not useful for the model at all
drop_cols = [
    'CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR',
    'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM', 'MEAN_ELEMENT_THEORY',
    'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'ELEMENT_SET_NO',
    'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2',
    'SITE', 'INTLDES', 'SATNAME', 'COMMENTCODE',
    'OBJECT_NUMBER', 'SOURCE', 'MSG_TYPE',
    'PRECEDENCE', 'type', 'id', 'attr_cosparId', 'attr_vimpelId',
    'attr_name', 'attr_mission', 'rel_launch', 'rel_reentry',
    'rel_initialOrbits', 'rel_destinationOrbits', 'rel_states',
    'rel_operators', 'rel_tags', 'rel_constellations',
    'OBJECT_ID', 'MSG_EPOCH', 'error', 'RA_OF_ASC_NODE',
    'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'MEAN_MOTION_DDOT',
    'REV_AT_EPOCH', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'LAUNCH',
    'COUNTRY_CODE', 'COUNTRY'
]

merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])
print("final shape:", merged.shape)
merged.head()

final shape: (66666, 54)


,OBJECT_NAME,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,NORAD_CAT_ID,BSTAR,MEAN_MOTION_DOT,SEMIMAJOR_AXIS,PERIOD,...,rx_km,ry_km,rz_km,vx_km_s,vy_km_s,vz_km_s,tle_age_days,altitude_km,speed_km_s,regime
0,OPS 3674 (VELA 4),2026-03-22T04:38:47.037408,0.23985689,0.57463450,48.7444,837,0.00000000000000,0.00000156,109420.923,6003.580,...,-87842.248216,66031.086790,40166.139083,-1.098975,-0.450464,1.326665,-4.0,110631.921857,1.780647,GEO
1,YZ-1 R/B,2026-03-22T00:05:53.344896,0.27910401,0.68547050,12.1133,41929,0.00000000000000,-0.00000606,98906.414,5159.367,...,-71978.947246,26862.548852,13005.642656,-2.142409,-1.165972,0.501345,-4.0,71550.191715,2.490131,GEO
2,ASTRON,2026-03-21T10:33:08.348544,0.24283977,0.65788428,78.4402,13901,0.00000000000000,0.00000203,108523.045,5929.836,...,-56540.136904,-127218.270086,-93797.684633,0.275027,0.036663,-0.999533,-3.0,161495.854880,1.037328,GEO
3,OPS 3662 (VELA 3),2026-03-20T19:37:22.087200,0.23902374,0.49573576,52.5257,836,0.00000000000000,0.00000640,109675.043,6024.506,...,-69574.444483,-75544.686826,128140.723117,0.671024,-0.859229,-0.177731,-3.0,157847.293602,1.104598,GEO
4,CXO,2026-03-20T11:08:23.080992,0.37808217,0.79854940,54.7244,25867,0.00000000000000,0.00001048,80787.538,3808.696,...,-25178.252625,-94524.345224,96227.317779,0.448764,-0.820913,0.023654,-2.0,130845.955703,0.935867,GEO


In [15]:
merged.to_parquet(f"{base}/merged/ML_merged.parquet", index=False)
print("saved!!", merged.shape)

saved!! (66666, 54)
